# PH Address Normalization & Fuzzy Matching Pipeline

**Part 1:** Normalize raw addresses → detect city (right-to-left) → filter `dim_location`  
**Part 2:** Fuzzy match to filtered `dim_location` → export `valid` / `invalid` Excel files

---

## 0. Install Dependencies

## 1. Imports & Configuration

Update the **file paths** and **`RAW_ADDRESSES`** list below to point at your actual data.

In [203]:
import os
import re
import unicodedata
import pandas as pd
from rapidfuzz import fuzz, process

# ── File Paths (update as needed) ─────────────────────────────────────────────
DIM_LOC_PATH = "../../data/mapping/dim_location_20260415_v3.csv"
ALIAS_PATH   = "../../data/utils/ph_address_alias_extended_v4.csv"
VALID_DIR    = "../../data/output/valid"
INVALID_DIR  = "../../data/output/invalid"

os.makedirs(VALID_DIR,   exist_ok=True)
os.makedirs(INVALID_DIR, exist_ok=True)

# ── Fuzzy match threshold (0–100) ─────────────────────────────────────────────
FUZZY_THRESHOLD = 70

print("Config OK ✓")

Config OK ✓


## 2. Raw Address Input

Replace the list below with your actual addresses, **or** load from a CSV/Excel file using the commented-out alternative.

In [204]:

# Load from file ──────────────────────────────────────────────────
psaddress_df    = pd.read_excel("../../data/sample_address.xlsx")
RAW_ADDRESSES = psaddress_df["order_deliveraddress"].dropna().tolist()

print(f"Loaded {len(RAW_ADDRESSES)} raw addresses")

Loaded 99 raw addresses


## 3. Load Reference Data

Loads `dim_location` (42k+ barangay records) and `ph_address_alias` (515 normalization rules).

In [205]:
print("Loading reference data …")

# dim_location
dim_raw = pd.read_csv(DIM_LOC_PATH, encoding="latin1")
dim_raw.columns = dim_raw.columns.str.strip()
for col in ["barangay_name", "city_name", "province_name", "region_name"]:
    dim_raw[col] = dim_raw[col].astype(str).str.strip()

# alias rules
alias_df = pd.read_csv(ALIAS_PATH, encoding="latin1", usecols=["pattern", "replacement"])
alias_df = alias_df.dropna(subset=["pattern", "replacement"])
alias_df["pattern"]     = alias_df["pattern"].astype(str).str.strip()
alias_df["replacement"] = alias_df["replacement"].astype(str).str.strip()
# Sort longest-first so longer patterns match before shorter ones
alias_df = alias_df.sort_values("pattern", key=lambda s: s.str.len(), ascending=False)
# Build lookup: UPPER pattern → UPPER replacement
alias_map = dict(zip(alias_df["pattern"].str.upper(), alias_df["replacement"].str.upper()))

print(f"  dim_location rows : {len(dim_raw):,}")
print(f"  alias rules       : {len(alias_map):,}")
dim_raw.head(3)

Loading reference data …
  dim_location rows : 42,011
  alias rules       : 520


,psgc_10_digit,region_name,province_name,city_name,barangay_name,region_code,province_code,city_code,barangay_code
0,1400101001,Cordillera Administrative Region (CAR),Abra,Bangued,Agtangao,14,1,1,1
1,1400101002,Cordillera Administrative Region (CAR),Abra,Bangued,Angad,14,1,1,2
2,1400101003,Cordillera Administrative Region (CAR),Abra,Bangued,Bañacao,14,1,1,3


**Reasoning**:
The subtask requires adding cleaned versions of 'city_name', 'province_name', and 'barangay_name' to the `dim_raw` DataFrame. I will apply the `clean_str` function to these columns and then display the head of the DataFrame to verify the new columns. This new cell will be inserted after `cell-load-ref` where `dim_raw` is loaded.



In [206]:
print("Adding cleaned columns to dim_raw...")
dim_raw["city_clean"] = dim_raw["city_name"].apply
dim_raw["prov_clean"] = dim_raw["province_name"].apply
dim_raw["bgy_clean"]  = dim_raw["barangay_name"].apply

print("Cleaned columns added to dim_raw ✓")
dim_raw.head()

Adding cleaned columns to dim_raw...
Cleaned columns added to dim_raw ✓


,psgc_10_digit,region_name,province_name,city_name,barangay_name,region_code,province_code,city_code,barangay_code,city_clean,prov_clean,bgy_clean
0,1400101001,Cordillera Administrative Region (CAR),Abra,Bangued,Agtangao,14,1,1,1,<bound method Series.apply of 0 Bangue...,<bound method Series.apply of 0 ...,<bound method Series.apply of 0 Agtan...
1,1400101002,Cordillera Administrative Region (CAR),Abra,Bangued,Angad,14,1,1,2,<bound method Series.apply of 0 Bangue...,<bound method Series.apply of 0 ...,<bound method Series.apply of 0 Agtan...
2,1400101003,Cordillera Administrative Region (CAR),Abra,Bangued,Bañacao,14,1,1,3,<bound method Series.apply of 0 Bangue...,<bound method Series.apply of 0 ...,<bound method Series.apply of 0 Agtan...
3,1400101004,Cordillera Administrative Region (CAR),Abra,Bangued,Bangbangar,14,1,1,4,<bound method Series.apply of 0 Bangue...,<bound method Series.apply of 0 ...,<bound method Series.apply of 0 Agtan...
4,1400101005,Cordillera Administrative Region (CAR),Abra,Bangued,Cabuloan,14,1,1,5,<bound method Series.apply of 0 Bangue...,<bound method Series.apply of 0 ...,<bound method Series.apply of 0 Agtan...


## 4. Helper Functions

In [207]:
def strip_accents(text: str) -> str:
    """Remove diacritical marks (e.g. ñ → n, é → e)."""
    return "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )

def clean_str(s: str) -> str:
    """Lowercase, strip accents, collapse whitespace."""
    s = strip_accents(str(s)).lower()
    return re.sub(r"\s+", " ", s).strip()

print("Helper functions defined ✓")

Helper functions defined ✓


**Reasoning**:
The subtask requires loading the `dictionary.json` file. I will import the `json` module, load the file into a variable, and then print the keys and a sample of its content to understand its structure.



In [208]:
import json

print("Loading blocking dictionary...")
BLOCKING_DICTIONARY_PATH = "../../data/dictionary/dictionary.json"

with open(BLOCKING_DICTIONARY_PATH, "r") as f:
    blocking_dictionary = json.load(f)

print(f"Blocking dictionary loaded with {len(blocking_dictionary):,} entries.")
print("First 5 keys:")
for i, key in enumerate(blocking_dictionary.keys()):
    if i >= 5:
        break
    print(f"- {key}")

print("\nSample entry for 'city_province_map':")
if "city_province_map" in blocking_dictionary:
    sample_map = {k: blocking_dictionary["city_province_map"][k] for k in list(blocking_dictionary["city_province_map"])[:3]}
    print(sample_map)
else:
    print("Key 'city_province_map' not found.")

Loading blocking dictionary...
Blocking dictionary loaded with 18 entries.
First 5 keys:
- Bangsamoro Autonomous Region In Muslim Mindanao (BARMM)
- Cordillera Administrative Region (CAR)
- MIMAROPA Region
- National Capital Region (NCR)
- Negros Island Region (NIR)

Sample entry for 'city_province_map':
Key 'city_province_map' not found.


**Reasoning**:
Before defining the `detect_city_improved` function, it's essential to prepare the necessary lookup dictionaries (`city_to_provinces`, `province_to_regions`, `all_cities_clean`) from the loaded `dim_raw` and `blocking_dictionary`. This ensures that all required data structures are available and correctly populated for the enhanced city detection logic. This step extracts the hierarchical relationships (region-province-city) from the `blocking_dictionary` and combines them with `dim_raw` data for comprehensive lookups.



In [209]:
# Helper to build city_to_provinces and province_to_regions
city_to_provinces = {}
province_to_regions = {}

for region_name, provinces_data in blocking_dictionary.items():
    clean_region = clean_str(region_name)
    for province_name, cities_data in provinces_data.items():
        clean_province = clean_str(province_name)
        province_to_regions.setdefault(clean_province, set()).add(clean_region)

        for city_name, city_aliases_data in cities_data["cities"].items():
            clean_city = clean_str(city_name)
            city_to_provinces.setdefault(clean_city, set()).add(clean_province)
            # Also add aliases as city keys pointing to the same province set
            for alias in city_aliases_data.get("aliases", []):
                clean_alias = clean_str(alias)
                city_to_provinces.setdefault(clean_alias, set()).add(clean_province)

# Recreate all_cities and all_cities_clean to ensure it's comprehensive and available
all_cities = dim_raw["city_name"].dropna().unique()
all_cities_clean = {clean_str(c): c for c in all_cities}   # clean → original
all_city_cleans = list(all_cities_clean.keys())

print("Blocking dictionary processed into city_to_provinces, province_to_regions, and all_cities_clean ✓")

Blocking dictionary processed into city_to_provinces, province_to_regions, and all_cities_clean ✓


---
## PART 1 — Normalize Addresses

Applies alias replacements token-by-token (up to 3-word patterns, longest-first).  
Examples: `BRGY.` → `BARANGAY`, `STA` → `SANTA`, `AVE` → `AVENUE`, `GENSAN` → `GENERAL SANTOS`

In [210]:
def normalize_address(addr: str) -> str:
    """
    Apply alias replacements word-by-word (and multi-word tokens, up to 3 words).
    Returns title-cased normalized string.
    """
    upper = addr.upper()
    tokens = re.split(r"(\s+|,|\.)", upper)
    result_tokens = []
    i = 0
    while i < len(tokens):
        tok = tokens[i].strip()
        if not tok or tok in (",", "."):
            result_tokens.append(tokens[i])
            i += 1
            continue
        matched = False
        for lookahead in (3, 2, 1):
            candidate_tokens = []
            j, count = i, 0
            while j < len(tokens) and count < lookahead:
                if re.match(r"^\s*$", tokens[j]) or tokens[j] in (",", "."):
                    j += 1
                    continue
                candidate_tokens.append(tokens[j])
                j += 1
                count += 1
            candidate = " ".join(candidate_tokens)
            if candidate in alias_map:
                result_tokens.append(alias_map[candidate])
                i = j
                matched = True
                break
        if not matched:
            result_tokens.append(tokens[i])
            i += 1
    normalized = re.sub(r"\s+", " ", "".join(result_tokens)).strip()
    return normalized.title()

print("normalize_address() defined ✓")

normalize_address() defined ✓


In [211]:
print("Normalizing addresses …")
records = [
    {"order_deliveraddress": addr, "normalized_address": normalize_address(addr)}
    for addr in RAW_ADDRESSES
]
normalized_df = pd.DataFrame(records)

print(f"Done — {len(normalized_df)} addresses normalized")
normalized_df

Normalizing addresses …
Done — 99 addresses normalized


,order_deliveraddress,normalized_address
0,Tagumpay sindalan CSFP,Tagumpay Sindalan San Fernando
1,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...","Block 16,Lot 4 Quiotan Street,Phil Am Life Vil..."
2,Cainta Rizal,Cainta Rizal
3,2079 a elias st sta cruz,2079 A Elias Street Santa Cruz
4,Swani Hardware 77 tirso neri street CDOC,Swani Hardware 77 Tirso Neri Street Cagayan De...
...,...,...
94,Phase 4C Package 7 Block 65 Lot 2 Brgy 176 Bag...,Phase 4C Package 7 Block 65 Lot 2 Barangay 176...
95,Blk 13 Lot 1 King St Exodus Floodway Taytay,Block 13 Lot 1 King Street Exodus Floodway Taytay
96,Blk 14 Lot 34 Imperial St. Manchester 11 exten...,Block 14 Lot 34 Imperial Street. Manchester 11...
97,Block 13 Lot 25 Buenavista Townhomes,Block 13 Lot 25 Buenavista Townhomes


### 1b. City Detection (Right-to-Left)

Reads each address from **right to left** (comma-delimited segments).  
Priority: exact word-boundary match → fuzzy `token_set_ratio ≥ 85`.

In [212]:
# Build city lookup from dim_location
all_cities       = dim_raw["city_name"].dropna().unique()
all_cities_clean = {clean_str(c): c for c in all_cities}   # clean → original
all_city_cleans  = list(all_cities_clean.keys())

def detect_city(norm_addr: str):
    """
    Right-to-left city detection across comma-delimited segments.
    Returns canonical city_name from dim_location, or None if not found.
    """
    segments_rtl = list(reversed([s.strip() for s in norm_addr.split(",")]))

    # 1) Exact word-boundary match (longest city wins)
    for seg in segments_rtl:
        seg_c = clean_str(seg)
        for city_c, city_orig in sorted(all_cities_clean.items(), key=lambda x: -len(x[0])):
            if re.search(r"\b" + re.escape(city_c) + r"\b", seg_c):
                return city_orig

    # 2) Fuzzy fallback per segment
    for seg in segments_rtl:
        seg_c = clean_str(seg)
        if len(seg_c) < 3:
            continue
        best = process.extractOne(seg_c, all_city_cleans, scorer=fuzz.token_set_ratio)
        if best and best[1] >= 75:
            return all_cities_clean[best[0]]

    return None

print("detect_city() defined ✓")

detect_city() defined ✓


**Reasoning**:
To fulfill the requirement of returning original casing for province names in the `detect_city_improved` function, I need to create a mapping from cleaned province names to their original forms. This mapping will be generated from the `province_name` column of the `dim_raw` DataFrame, ensuring that the canonical province names retain their original casing.



In [213]:
print("Preparing province clean to original mapping...")
province_clean_to_original = {clean_str(p): p for p in dim_raw["province_name"].dropna().unique()}
print("Province clean to original mapping ready ✓")

def detect_city_improved(norm_addr: str) -> list[tuple[str, str]]:
    """
    Right-to-left city detection across comma-delimited segments.
    Returns a list of (canonical_city_name, canonical_province_name) tuples if found,
    or an empty list if no city is detected.
    """
    candidate_pairs = set()  # Use set to store unique (city, province) candidates
    segments_rtl = list(reversed([s.strip() for s in norm_addr.split(",")]))

    for seg in segments_rtl:
        seg_c = clean_str(seg)
        if len(seg_c) < 3: # Skip very short segments for efficiency
            continue

        # 1) Exact word-boundary match (longest city wins)
        matched_city_found_in_segment = False
        for city_c, city_orig in sorted(all_cities_clean.items(), key=lambda x: -len(x[0])):
            if re.search(r"\b" + re.escape(city_c) + r"\b", seg_c):
                provinces_for_city = city_to_provinces.get(city_c, set())
                for prov_clean in provinces_for_city:
                    original_prov = province_clean_to_original.get(prov_clean)
                    if original_prov:
                        candidate_pairs.add((city_orig, original_prov))
                matched_city_found_in_segment = True
                break # Break after first exact match in segment (longest one)

        if matched_city_found_in_segment:
            continue # Move to next segment if exact match found

        # 2) Fuzzy fallback per segment if no exact match found
        best = process.extractOne(seg_c, all_city_cleans, scorer=fuzz.token_set_ratio)
        if best and best[1] >= 85:
            matched_city_clean = best[0]
            matched_city_original = all_cities_clean[matched_city_clean]
            provinces_for_city = city_to_provinces.get(matched_city_clean, set())
            for prov_clean in provinces_for_city:
                original_prov = province_clean_to_original.get(prov_clean)
                if original_prov:
                    candidate_pairs.add((matched_city_original, original_prov))

    return list(candidate_pairs)

print("detect_city_improved() defined ✓")

Preparing province clean to original mapping...
Province clean to original mapping ready ✓
detect_city_improved() defined ✓


In [214]:
print("Detecting cities …")
normalized_df["detected_city"] = normalized_df["normalized_address"].apply(detect_city)

has_city = normalized_df[normalized_df["detected_city"].notna()].copy()
no_city  = normalized_df[normalized_df["detected_city"].isna()].copy()

print(f"  ✅ Addresses with city    : {len(has_city)}")
print(f"  ❌ Addresses without city : {len(no_city)} → invalid")
normalized_df[["order_deliveraddress", "normalized_address", "detected_city"]]

Detecting cities …
  ✅ Addresses with city    : 75
  ❌ Addresses without city : 24 → invalid


,order_deliveraddress,normalized_address,detected_city
0,Tagumpay sindalan CSFP,Tagumpay Sindalan San Fernando,San Fernando
1,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...","Block 16,Lot 4 Quiotan Street,Phil Am Life Vil...",Carmen
2,Cainta Rizal,Cainta Rizal,Cainta
3,2079 a elias st sta cruz,2079 A Elias Street Santa Cruz,Santa Cruz
4,Swani Hardware 77 tirso neri street CDOC,Swani Hardware 77 Tirso Neri Street Cagayan De...,Cagayan De Oro
...,...,...,...
94,Phase 4C Package 7 Block 65 Lot 2 Brgy 176 Bag...,Phase 4C Package 7 Block 65 Lot 2 Barangay 176...,Caloocan
95,Blk 13 Lot 1 King St Exodus Floodway Taytay,Block 13 Lot 1 King Street Exodus Floodway Taytay,Taytay
96,Blk 14 Lot 34 Imperial St. Manchester 11 exten...,Block 14 Lot 34 Imperial Street. Manchester 11...,NaN
97,Block 13 Lot 25 Buenavista Townhomes,Block 13 Lot 25 Buenavista Townhomes,Buenavista


### 1c. Filter dim_location to Detected Cities

Reduces the 42k-row reference table to only the cities present in the address batch — speeds up fuzzy matching significantly.

In [215]:
detected_cities = has_city["detected_city"].dropna().unique()
dim_filtered    = dim_raw[dim_raw["city_name"].isin(detected_cities)].copy()

# Pre-clean columns for matching
dim_filtered["city_clean"] = dim_filtered["city_name"].apply(clean_str)
dim_filtered["prov_clean"] = dim_filtered["province_name"].apply(clean_str)
dim_filtered["bgy_clean"]  = dim_filtered["barangay_name"].apply(clean_str)

print(f"Filtered dim_location: {len(dim_raw):,} → {len(dim_filtered):,} rows")
print(f"Cities included: {list(detected_cities)}")
dim_filtered.head(5)

Filtered dim_location: 42,011 → 3,752 rows
Cities included: ['San Fernando', 'Carmen', 'Cainta', 'Santa Cruz', 'Cagayan De Oro', 'Tapaz', 'Santa', 'Enrile', 'Legazpi', 'Calinog', 'Santa Rosa', 'Concepcion', 'Bulacan', 'Santa Ana', 'San Isidro', 'San Pablo', 'Binalonan', 'Sampaloc', 'Las Piñas', 'Rosario', 'Famy', 'San Mateo', 'Rizal', 'Tanza', 'Valladolid', 'San Juan', 'Bago', 'Puerto Princesa', 'Cadiz', 'Dasmariñas', 'Cavite', 'Santa Teresita', 'Valenzuela', 'San Pedro', 'San Felipe', 'Bacolor', 'Lipa', 'Malay', 'Taft', 'Parañaque', 'Daet', 'San Agustin', 'Quezon', 'San Francisco', 'Pakil', 'Mangaldan', 'Socorro', 'Liloan', 'Binondo', 'Talisay', 'Pila', 'Banaybanay', 'Caloocan', 'Taytay', 'Buenavista', 'Mexico']


,psgc_10_digit,region_name,province_name,city_name,barangay_name,region_code,province_code,city_code,barangay_code,city_clean,prov_clean,bgy_clean
230,1400121001,Cordillera Administrative Region (CAR),Abra,San Isidro,Cabayogan,14,1,21,1,san isidro,abra,cabayogan
231,1400121002,Cordillera Administrative Region (CAR),Abra,San Isidro,Dalimag,14,1,21,2,san isidro,abra,dalimag
232,1400121003,Cordillera Administrative Region (CAR),Abra,San Isidro,Langbaban,14,1,21,3,san isidro,abra,langbaban
233,1400121004,Cordillera Administrative Region (CAR),Abra,San Isidro,Manayday,14,1,21,4,san isidro,abra,manayday
234,1400121007,Cordillera Administrative Region (CAR),Abra,San Isidro,Pantoc,14,1,21,7,san isidro,abra,pantoc


---
## PART 2 — Fuzzy Matching to dim_location

For each address:
1. Sub-filters `dim_location` to the detected city
2. Fuzzy-matches the full address string against barangay names (`partial_ratio ≥ FUZZY_THRESHOLD`)
3. Computes `city_match`, `province_match`, `city_match_fuzzy`, `province_match_fuzzy` flags

In [216]:
def fuzzy_match_flag(a: str, b: str, threshold: int = FUZZY_THRESHOLD) -> bool:
    """Returns True if partial_ratio(a, b) >= threshold."""
    return fuzz.partial_ratio(str(a), str(b)) >= threshold

def match_address(row) -> dict:
    """
    Matches a single normalized address row to dim_location.
    Returns a dict of all output columns.
    """
    norm     = row["normalized_address"]
    det_city = row["detected_city"]
    addr_c   = clean_str(norm)

    sub = dim_filtered[dim_filtered["city_name"] == det_city].copy()

    if sub.empty:
        return {
            "barangay_code": None, "barangay_name": None,
            "city_code": None,     "city_name": det_city,
            "province_code": None, "province_name": None,
            "region_code": None,   "region_name": None,
            "addr_clean": addr_c,
            "city_clean": clean_str(det_city), "prov_clean": "",
            "city_match": False, "province_match": False,
            "city_match_fuzzy": False, "province_match_fuzzy": False,
            "match_status": "city_only",
        }

    city_c = clean_str(det_city)
    prov_c = clean_str(sub["province_name"].iloc[0])

    # Barangay fuzzy match within city
    bgy_names  = sub["bgy_clean"].tolist()
    best_match = process.extractOne(addr_c, bgy_names, scorer=fuzz.partial_ratio)

    if best_match and best_match[1] >= FUZZY_THRESHOLD:
        matched_row  = sub[sub["bgy_clean"] == best_match[0]].iloc[0]
        match_status = "valid"
        barangay_code = matched_row["barangay_code"]
        barangay_name = matched_row["barangay_name"]
    else:
        matched_row  = sub.iloc[0]   # city-level fallback
        match_status = "city_only"
        barangay_code = None
        barangay_name = None

    return {
        "barangay_code":        barangay_code,
        "barangay_name":        barangay_name,
        "city_code":            matched_row["city_code"],
        "city_name":            matched_row["city_name"],
        "province_code":        matched_row["province_code"],
        "province_name":        matched_row["province_name"],
        "region_code":          matched_row["region_code"],
        "region_name":          matched_row["region_name"],
        "addr_clean":           addr_c,
        "city_clean":           city_c,
        "prov_clean":           prov_c,
        "city_match":           city_c in addr_c,
        "province_match":       prov_c in addr_c,
        "city_match_fuzzy":     fuzzy_match_flag(city_c, addr_c),
        "province_match_fuzzy": fuzzy_match_flag(prov_c, addr_c),
        "match_status":         match_status,
    }

print("match_address() defined ✓")

match_address() defined ✓


In [217]:
print("Running fuzzy matching …")
matched_results = has_city.apply(match_address, axis=1, result_type="expand")
valid_df = pd.concat(
    [has_city.reset_index(drop=True), matched_results.reset_index(drop=True)],
    axis=1
)

print(f"Matching complete — {len(valid_df)} addresses processed")
valid_df[["order_deliveraddress", "city_name", "province_name", "barangay_name",
          "city_match", "city_match_fuzzy", "province_match_fuzzy", "match_status"]]

Running fuzzy matching …
Matching complete — 75 addresses processed


,order_deliveraddress,city_name,province_name,barangay_name,city_match,city_match_fuzzy,province_match_fuzzy,match_status
0,Tagumpay sindalan CSFP,San Fernando,Pampanga,NaN,True,True,False,valid
1,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...",Carmen,Surigao del Sur,Carmen,True,True,False,valid
2,Cainta Rizal,Cainta,Rizal,Santa Rosa,True,True,True,valid
3,2079 a elias st sta cruz,Santa Cruz,Laguna,Santisima Cruz,True,True,False,valid
4,Swani Hardware 77 tirso neri street CDOC,Cagayan De Oro,Misamis Oriental,NaN,True,True,False,city_only
...,...,...,...,...,...,...,...,...
70,"BRGY. CONCEPCION,BANAYBANAY",Banaybanay,Davao Oriental,Rang-ay,True,True,False,valid
71,Phase 4C Package 7 Block 65 Lot 2 Brgy 176 Bag...,Caloocan,Metro Manila,Barangay 1,True,True,False,valid
72,Blk 13 Lot 1 King St Exodus Floodway Taytay,Taytay,Palawan,NaN,True,True,False,city_only
73,Block 13 Lot 25 Buenavista Townhomes,Buenavista,Quezon,Buenavista,True,True,False,valid


## 5. Segregate Valid / Invalid

In [218]:
OUTPUT_COLS = [
    "order_deliveraddress", "normalized_address",
    "barangay_code", "barangay_name",
    "city_code",     "city_name",
    "province_code", "province_name",
    "region_code",   "region_name",
    "addr_clean",    "city_clean",  "prov_clean",
    "city_match",    "province_match",
    "city_match_fuzzy", "province_match_fuzzy",
]

def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = None
    return df[cols]

valid_final   = valid_df[valid_df["match_status"] == "valid"].copy()
invalid_part2 = valid_df[valid_df["match_status"] != "valid"].copy()
invalid_final = pd.concat(
    [no_city.assign(match_status="no_city_detected"), invalid_part2],
    ignore_index=True
)

valid_out   = ensure_cols(valid_final.copy(),   OUTPUT_COLS)
invalid_out = ensure_cols(invalid_final.copy(), OUTPUT_COLS)

print(f"✅ Valid   : {len(valid_out)} addresses")
print(f"❌ Invalid : {len(invalid_out)} addresses")

✅ Valid   : 54 addresses


❌ Invalid : 45 addresses


### Preview Results

In [219]:
print("=== VALID ===")
valid_out

=== VALID ===


,order_deliveraddress,normalized_address,barangay_code,barangay_name,city_code,city_name,province_code,province_name,region_code,region_name,addr_clean,city_clean,prov_clean,city_match,province_match,city_match_fuzzy,province_match_fuzzy
0,Tagumpay sindalan CSFP,Tagumpay Sindalan San Fernando,21.0,NaN,16,San Fernando,54,Pampanga,3,Region III (Central Luzon),tagumpay sindalan san fernando,san fernando,bukidnon,True,False,True,False
1,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...","Block 16,Lot 4 Quiotan Street,Phil Am Life Vil...",3.0,Carmen,6,Carmen,68,Surigao del Sur,16,Region XIII (Caraga),"block 16,lot 4 quiotan street,phil am life vil...",carmen,agusan del norte,True,False,True,False
2,Cainta Rizal,Cainta Rizal,18.0,Santa Rosa,5,Cainta,58,Rizal,4,Region IV-A (CALABARZON),cainta rizal,cainta,rizal,True,True,True,True
3,2079 a elias st sta cruz,2079 A Elias Street Santa Cruz,23.0,Santisima Cruz,26,Santa Cruz,34,Laguna,4,Region IV-A (CALABARZON),2079 a elias street santa cruz,santa cruz,davao del sur,True,False,True,False
6,4662PagAsa St V mapa Sta Mesa Manila,4662Pagasa Street V Mapa Santa Mesa Manila,1.0,Ampandula,22,Santa,29,Ilocos Sur,1,Region I (Ilocos Region),4662pagasa street v mapa santa mesa manila,santa,ilocos sur,True,False,True,False
8,873RizalStAlbayDistrict Bgy. 17 - Rizal Sreet....,873Rizalstalbaydistrict Barangay. 17 - Rizal S...,13.0,"Bgy. 17 - Rizal Street., Ilawod",6,Legazpi,5,Albay,5,Region V (Bicol Region),873rizalstalbaydistrict barangay. 17 - rizal s...,legazpi,albay,True,True,True,True
9,Barrio Calinog Iloilo,Barangay Calinog Iloilo City,32.0,Garangan,13,Calinog,30,Iloilo,6,Region VI (Western Visayas),barangay calinog iloilo city,calinog,iloilo,True,True,True,True
10,"3101 DEEPWELL STOROSARIO,STA ROSA NUEVA ECIJA","3101 Deepwell Storosario,Santa Rosa Nueva Ecija",18.0,Santo Rosario,28,Santa Rosa,49,Nueva Ecija,3,Region III (Central Luzon),"3101 deepwell storosario,santa rosa nueva ecija",santa rosa,laguna,True,False,True,False
13,2161 Onyx St Santa Ana Manila,2161 Onyx Street Santa Ana Manila,12.0,Santa Maria,19,Santa Ana,54,Pampanga,3,Region III (Central Luzon),2161 onyx street santa ana manila,santa ana,cagayan,True,False,True,False
15,BIR Bulua CDO,Bir Bulua Cagayan De Oro,3.0,Bulua,0,Cagayan De Oro,43,Misamis Oriental,10,Region X (Northern Mindanao),bir bulua cagayan de oro,cagayan de oro,misamis oriental,True,False,True,False


In [220]:
print("=== INVALID ===")
invalid_out

=== INVALID ===


,order_deliveraddress,normalized_address,barangay_code,barangay_name,city_code,city_name,province_code,province_name,region_code,region_name,addr_clean,city_clean,prov_clean,city_match,province_match,city_match_fuzzy,province_match_fuzzy
0,ADP Bldg Gov. A Lugay Ave Mangga Dos Matatalai...,Adp Building Gov. A Lugay Avenue Mangga Dos Ma...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,tayuman manila,Tayuman Manila,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CAGAYAN 3500 GOMEZ STREET,Cagayan 3500 Gomez Street,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ANA ROS VILLAGE MANDURRIAO,Ana Ros Village Mandurriao,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,19 Tamblot St Proj 4,19 Tamblot Street Proj 4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Brgy cabanutuan nueva ecija,Barangay Cabanutuan Nueva Ecija,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,"Gahonon, DCN","Gahonon, Camarines Norte",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,B2 Claudio Street Estacio Village,B2 Claudio Street Estacio Village,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,BRGY FONZALES TANUAN CITY BATNGAS,Barangay Fonzales Tanuan City Batngas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,"REBOLLOS DRIVE, PORCENTRO","Rebollos Drive, Porcentro",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Export to Excel

In [221]:
from openpyxl.styles import Font, PatternFill, Alignment

def write_excel(df: pd.DataFrame, path: str, sheet: str = "Results"):
    """Write DataFrame to a styled Excel file with frozen header row."""
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name=sheet)
        ws = writer.sheets[sheet]
        # Auto-fit column widths
        for col_cells in ws.columns:
            max_len = max((len(str(c.value)) if c.value else 0) for c in col_cells)
            ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 4, 60)
        # Header style
        header_fill = PatternFill("solid", fgColor="1F4E79")
        for cell in ws[1]:
            cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
            cell.fill      = header_fill
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        ws.row_dimensions[1].height = 30
        ws.freeze_panes = "A2"
    print(f"  Exported → {path}")

from datetime import datetime

export_date = datetime.now().strftime("%Y%m%d")
valid_path = os.path.join(VALID_DIR, f"valid_addresses_{export_date}.xlsx")
invalid_path = os.path.join(INVALID_DIR, f"invalid_addresses_{export_date}.xlsx")

write_excel(valid_out, valid_path, "Valid")
write_excel(invalid_out, invalid_path, "Invalid")

print("\nPipeline complete ✓")
print(f" Valid → {valid_path}")
print(f" Invalid → {invalid_path}")

  Exported → ../../data/output/valid\valid_addresses_20260416.xlsx
  Exported → ../../data/output/invalid\invalid_addresses_20260416.xlsx

Pipeline complete ✓
 Valid → ../../data/output/valid\valid_addresses_20260416.xlsx
 Invalid → ../../data/output/invalid\invalid_addresses_20260416.xlsx


# Task
Refine the address normalization and fuzzy matching pipeline to improve accuracy for ambiguous addresses, particularly for cases like 'Tagumpay Sindalan CSFP'. This involves loading and incorporating additional data from `dictionary.json` for enhanced location lookups, updating the `detect_city` function to generate city-province candidates when initial detection is ambiguous, and modifying the `match_address` function to prioritize barangay matching and iteratively disambiguate using city variants. Finally, re-run the updated pipeline and summarize the improvements, remaining challenges, and updated counts of valid and invalid addresses.

## Load and Prepare Blocking Dictionary

### Subtask:
Load the `dictionary.json` file and process its content to create lookup structures for enhanced location lookups and disambiguation.


# Task
Prepare reference data by adding cleaned versions of 'city_name', 'province_name', and 'barangay_name' as new columns ('city_clean', 'prov_clean', 'bgy_clean') to the `dim_raw` DataFrame.

## Prepare Reference Data with Cleaned Columns

### Subtask:
Add cleaned versions of 'city_name', 'province_name', and 'barangay_name' as new columns ('city_clean', 'prov_clean', 'bgy_clean') to the `dim_raw` DataFrame.


## Refine City Detection to Return Candidates

### Subtask:
Modify the `detect_city` function to leverage the `city_to_provinces` mapping, returning a list of `(canonical_city_name, canonical_province_name)` tuples for detected cities.
